# Module 6 · Pipelines de données — Travaux pratiques
**Togo AI Lab · Formation IA · Version IT** · Durée : environ 1 h 30

**Ce que vous allez faire :** construire avec **pandas** un petit pipeline qui transforme un registre de vaccination **fictif et très sale** (300 fiches, doublons, formats mélangés, variantes d'écriture, valeurs impossibles) en données fiables : profilage, nettoyage, validation avec mise en quarantaine, pseudonymisation, contrôle de k-anonymat, version et rapport de qualité.

**À la fin, vous saurez :**
1. profiler un jeu de données et chiffrer ses défauts ;
2. écrire des étapes de nettoyage **simples, testables et rejouables** (idempotentes) ;
3. exprimer des règles de validation et isoler les lignes en défaut ;
4. pseudonymiser un identifiant et vérifier le risque de ré-identification (k-anonymat) ;
5. assembler un pipeline avec journal, version et rapport de qualité.

**Comment utiliser ce notebook (Google Colab)**
- Exécutez les cellules avec **Maj + Entrée**, dans l'ordre. Aucun GPU n'est nécessaire.
- Les cellules **👉 À vous** contiennent des `____` à remplacer. Toutes les données sont **fictives**.

## 0 · Préparation

In [ ]:
import re, json, hashlib, unicodedata
import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
rng = np.random.default_rng(42)
print("Outils prêts ✔ — pandas", pd.__version__)

## 1 · Le registre sale
Un **générateur** fabrique d'abord un registre **propre** (la vérité), puis y injecte des défauts réalistes. Nous garderons la vérité pour mesurer notre nettoyage. **Exécutez la cellule sans la lire en détail.**

In [ ]:
# --- Générateur (ne pas lire en détail) ---
LOCALITES = ["Kélé", "Kara", "Sokodé", "Atakpamé", "Dapaong"]
VACCINS   = ["ROR", "Penta", "BCG", "Polio"]
SEXES     = ["F", "M"]
N = 300

def sans_accents(t):
    return "".join(c for c in unicodedata.normalize("NFD", str(t)) if unicodedata.category(c) != "Mn")

naissance = pd.to_datetime("2021-01-01") + pd.to_timedelta(rng.integers(0, 1640, N), unit="D")
delai = pd.to_timedelta(rng.integers(60, 900, N), unit="D")
fin = pd.to_datetime("2026-02-28")
date_vac = np.minimum(naissance + delai, fin)
propre = pd.DataFrame({
    "id_fiche": [f"F{i:04d}" for i in range(1, N + 1)],
    "nom": [f"Enfant {i:03d}" for i in range(1, N + 1)],
    "naissance": naissance, "sexe": rng.choice(SEXES, N),
    "localite": rng.choice(LOCALITES, N), "vaccin": rng.choice(VACCINS, N),
    "date_vaccin": pd.to_datetime(date_vac)})

sale = propre.copy()
for c in ("naissance", "date_vaccin"):
    sale[c] = sale[c].dt.strftime("%d/%m/%Y")
def tirer(p): return rng.random(N) < p
m = tirer(0.12); sale.loc[m, "naissance"] = propre.loc[m, "naissance"].dt.strftime("%Y-%m-%d")
m = tirer(0.08); sale.loc[m, "localite"] = sale.loc[m, "localite"].str.upper()
m = tirer(0.05); sale.loc[m, "localite"] = sale.loc[m, "localite"].map(sans_accents)
m = tirer(0.04); sale.loc[m, "localite"] = sale.loc[m, "localite"] + " "
m = tirer(0.08); sale.loc[m, "vaccin"] = sale.loc[m, "vaccin"].str.lower()
m = tirer(0.05); sale.loc[m, "sexe"] = sale.loc[m, "sexe"].str.lower()
m = tirer(0.03); sale.loc[m, "naissance"] = ""
m = tirer(0.02); sale.loc[m, "date_vaccin"] = "31/02/2026"
m = tirer(0.02); sale.loc[m, "naissance"] = "10/10/2030"
m = tirer(0.02); sale.loc[m, "date_vaccin"] = "01/01/2019"
m = tirer(0.01); sale.loc[m, "sexe"] = "X"
sale = pd.concat([sale, sale.sample(15, random_state=1)]).sample(frac=1, random_state=2).reset_index(drop=True)
print(sale.shape, "lignes (dont 15 doublons injectés)")
sale.head(8)

## 2 · Profiler : chiffrer les défauts
Avant de nettoyer, on **mesure**. Un bon profil répond à : combien de valeurs manquantes ? combien de doublons ? quelles variantes d'écriture ? quels formats de date ?

In [ ]:
def forme(s):
    """Forme d'une valeur : chiffres → 9, lettres → a (pour repérer les formats)."""
    return re.sub(r"[A-Za-zÀ-ÿ]", "a", re.sub(r"\d", "9", str(s)))

print("Valeurs vides par colonne :")
print((sale.replace("", np.nan).isna().sum()).to_string())
print("\nDoublons exacts :", int(sale.duplicated().sum()))
print("\nFormes de la colonne naissance :")
print(sale["naissance"].map(forme).value_counts().to_string())
print("\nLocalités :", sorted(sale["localite"].unique()))
print("Vaccins   :", sorted(sale["vaccin"].unique()))
print("Sexes     :", sorted(sale["sexe"].unique()))

## 3 · Nettoyer, une étape à la fois
Chaque étape est une **fonction pure** : elle reçoit un tableau et en renvoie un **nouveau** (jamais de modification du brut). C'est ce qui permet de tester chaque étape et de la rejouer.

### 3.1 Dates
Deux formats coexistent : `JJ/MM/AAAA` et `AAAA-MM-JJ`. On essaie chaque format, `errors="coerce"` transforme ce qui est illisible en `NaT` (qu'on comptera et signalera au lieu de l'ignorer).

👉 **À vous (1/7)** : complétez le format de la première tentative.

In [ ]:
def parser_date(serie):
    a = pd.to_datetime(serie, format="%d/%m/%Y", errors="coerce")
    b = pd.to_datetime(serie, format="%Y-%m-%d", errors="coerce")
    return a.combine_first(b)

def etape_dates(df):
    df = df.copy()
    for c in ("naissance", "date_vaccin"):
        df[c] = parser_date(df[c])
    return df

t = etape_dates(sale)
print(t[["naissance", "date_vaccin"]].dtypes.to_string())
print("Dates illisibles (NaT) : naissance =", int(t["naissance"].isna().sum()), "| date_vaccin =", int(t["date_vaccin"].isna().sum()))

**Question :** les « 31/02/2026 » sont devenus `NaT`, ainsi que les champs vides. Pourquoi est-ce préférable à une date inventée ? Comment distinguer ensuite « vide » et « impossible » ?

### 3.2 Catégories
On ramène chaque variante à la **valeur de référence** : on enlève espaces, accents et casse pour comparer, puis on remplace par la forme officielle. Ce qui n'est pas dans la liste reste vide (et sera signalé).

👉 **À vous (2/7)** : complétez la clé de comparaison (sans accents, en minuscules).

In [ ]:
def referentiel(valeurs):
    return {sans_accents(v).lower(): v for v in valeurs}

def normaliser_colonne(serie, valeurs_ref):
    ref = referentiel(valeurs_ref)
    cle = serie.astype(str).str.strip().map(sans_accents).str.lower()
    return cle.map(ref)          # NaN si hors liste

def etape_categories(df):
    df = df.copy()
    df["localite"] = normaliser_colonne(df["localite"], LOCALITES)
    df["vaccin"] = normaliser_colonne(df["vaccin"], VACCINS)
    df["sexe"] = normaliser_colonne(df["sexe"], SEXES)
    return df

t = etape_categories(sale)
print("Localités :", sorted(t["localite"].dropna().unique()))
print("Sexes hors liste (devenus vides) :", int(t["sexe"].isna().sum()))

### 3.3 Doublons
Un doublon se définit **par une clé métier**, pas par « toute la ligne identique ». Ici : même fiche, même vaccin, même date.

👉 **À vous (3/7)** : indiquez les colonnes qui définissent un doublon.

In [ ]:
CLE = ["id_fiche", "vaccin", "date_vaccin"]

def etape_doublons(df):
    return df.drop_duplicates(subset=CLE, keep="first")

avant = len(sale)
apres = len(etape_doublons(etape_categories(etape_dates(sale))))
print(f"{avant} lignes → {apres} lignes ({avant - apres} doublons retirés)")

**Attention :** après avoir normalisé les valeurs (3.1 et 3.2), on détecte **plus** de doublons qu'avant. L'**ordre** des étapes compte : normaliser avant de dédoublonner.

## 4 · Valider : des règles, pas des impressions
Une **règle de validation** est une fonction qui renvoie, pour chaque ligne, vrai (en défaut) ou faux. On ne corrige pas à l'aveugle : on **isole** les lignes en défaut dans une **quarantaine**, avec le motif.

👉 **À vous (4/7)** : complétez la règle d'incohérence : un vaccin ne peut pas être antérieur à la naissance.

In [ ]:
AUJOURDHUI = pd.Timestamp("2026-03-01")

def regles(df):
    return {
        "naissance_manquante":     df["naissance"].isna(),
        "date_vaccin_manquante":   df["date_vaccin"].isna(),
        "naissance_future":        df["naissance"] > AUJOURDHUI,
        "vaccin_avant_naissance":  df["date_vaccin"] < df["naissance"],
        "localite_hors_liste":     df["localite"].isna(),
        "vaccin_hors_liste":       df["vaccin"].isna(),
        "sexe_hors_liste":         df["sexe"].isna(),
    }

def valider(df):
    r = regles(df)
    motifs = pd.DataFrame(r)
    df = df.copy()
    df["motifs"] = motifs.apply(lambda ligne: ",".join(ligne.index[ligne]), axis=1)
    valides = df[df["motifs"] == ""].drop(columns="motifs")
    quarantaine = df[df["motifs"] != ""]
    return valides, quarantaine

net = etape_doublons(etape_categories(etape_dates(sale)))
valides, quarantaine = valider(net)
print(len(valides), "lignes valides |", len(quarantaine), "en quarantaine")
print(pd.DataFrame(regles(net)).sum().rename("lignes en défaut").to_string())

In [ ]:
quarantaine[["id_fiche", "naissance", "sexe", "localite", "vaccin", "date_vaccin", "motifs"]].head(10)

**À observer :** la quarantaine contient le **motif** de chaque rejet : c'est ce qu'on renvoie à l'agent de collecte pour corriger à la source. Rejeter sans motif empêche d'améliorer la collecte.

### Mesurer le nettoyage
Comme nous connaissons la vérité (`propre`), mesurons la part des lignes **valides** qui coïncident exactement avec elle.

In [ ]:
ref = propre.set_index("id_fiche")
v = valides.set_index("id_fiche")
colonnes = ["naissance", "sexe", "localite", "vaccin", "date_vaccin"]
commun = v.index.intersection(ref.index)
exact = (v.loc[commun, colonnes] == ref.loc[commun, colonnes]).all(axis=1)
print(f"{len(commun)} fiches valides, {exact.mean():.1%} identiques à la vérité terrain")
print("Fiches de la vérité absentes du résultat :", len(ref.index.difference(v.index)), "(en quarantaine ou perdues)")

**Question :** les fiches sont-elles « perdues » ou en quarantaine ? Qu'adviendrait-il des chiffres de couverture vaccinale si on les supprimait silencieusement ?

## 5 · Protéger : pseudonymiser et regrouper
Pour un jeu destiné à l'analyse, on remplace l'identifiant par un **pseudonyme** (haché avec un **sel** secret), on retire le nom, et on **généralise** la date de naissance.

👉 **À vous (5/7)** : complétez le hachage SHA-256 (le sel est concaténé à l'identifiant). 

In [ ]:
SEL = "sel-de-demonstration"   # En production : secret, hors du code (gestionnaire de secrets).

def pseudonyme(valeur):
    return hashlib.sha256((SEL + str(valeur)).encode("utf-8")).hexdigest()[:10]

def etape_protection(df):
    df = df.copy()
    df["pseudo"] = df["id_fiche"].map(pseudonyme)
    df["annee_naissance"] = df["naissance"].dt.year
    return df.drop(columns=["id_fiche", "nom", "naissance"])

protege = etape_protection(valides)
print(protege.head(5).to_string())
print("Même entrée, même pseudonyme ?", pseudonyme("F0001") == pseudonyme("F0001"))

**À retenir :** un identifiant haché est **pseudonymisé, pas anonymisé** : qui possède le sel (ou peut essayer toutes les valeurs possibles) peut le retrouver. Les données restent des **données personnelles**.

### Contrôler le risque : le k-anonymat
Un jeu est **k-anonyme** si chaque combinaison de **quasi-identifiants** (ici : année de naissance, sexe, localité) concerne au moins **k** personnes.

👉 **À vous (6/7)** : calculez la taille de chaque groupe, puis la taille du plus petit.

In [ ]:
QUASI = ["annee_naissance", "sexe", "localite"]

def k_anonymat(df, quasi=QUASI):
    groupes = df.groupby(quasi).size()
    return int(groupes.min()), int((groupes < 5).sum()), len(groupes)

k, petits, total = k_anonymat(protege)
print(f"k = {k} ; groupes de moins de 5 personnes : {petits} sur {total}")

In [ ]:
# Généraliser : tranche de deux ans plutôt que l'année, et on retire le sexe pour le rapport public
protege2 = protege.copy()
protege2["periode_naissance"] = (protege2["annee_naissance"] // 2 * 2).astype(str) + "-" + (protege2["annee_naissance"] // 2 * 2 + 1).astype(str)
k2, petits2, total2 = k_anonymat(protege2, ["periode_naissance", "localite"])
print(f"Après généralisation : k = {k2} ; groupes < 5 : {petits2} sur {total2}")

**À observer :** regrouper fait monter `k` (moins de risque) mais réduit la finesse de l'analyse : c'est un **compromis utilité / confidentialité**. Le bon niveau dépend de l'usage et du cadre légal.

## 6 · Assembler le pipeline
On compose les étapes, avec un **journal** (nombre de lignes à chaque étape), une **quarantaine** et un test d'**idempotence** : rejouer le pipeline sur le même brut doit donner exactement le même résultat.

👉 **À vous (7/7)** : complétez la boucle qui applique les étapes dans l'ordre.

In [ ]:
ETAPES = [("dates", etape_dates), ("catégories", etape_categories), ("doublons", etape_doublons)]

def pipeline(brut):
    journal = [("brut", len(brut))]
    df = brut
    for nom, etape in ETAPES:
        df = etape(df)
        journal.append((nom, len(df)))
    valides, quarantaine = valider(df)
    journal += [("valides", len(valides)), ("quarantaine", len(quarantaine))]
    return valides.reset_index(drop=True), quarantaine.reset_index(drop=True), journal

v1, q1, journal = pipeline(sale)
v2, q2, _ = pipeline(sale)
print(pd.DataFrame(journal, columns=["étape", "lignes"]).to_string(index=False))
print("\nIdempotent (deux exécutions identiques) :", v1.equals(v2) and q1.equals(q2))

### Nouveau lot : ajouter sans dupliquer
Chaque semaine arrive un nouveau lot. Le pipeline doit **fusionner** sans créer de doublons, même si un lot est **rejoué** par erreur.

In [ ]:
entrepot = v1.copy()
def ajouter_lot(entrepot, lot_brut):
    v, q, _ = pipeline(lot_brut)
    fusion = pd.concat([entrepot, v]).drop_duplicates(subset=CLE, keep="first")
    return fusion.reset_index(drop=True), q

lot = sale.sample(40, random_state=7)          # un lot déjà vu, rejoué par erreur
apres, q = ajouter_lot(entrepot, lot)
print("Avant :", len(entrepot), "| après rejeu du même lot :", len(apres), "| identique :", apres.equals(entrepot))

## 7 · Version, rapport de qualité et alerte
Deux garde-fous de production :
- une **version** du jeu produit (date, nombre de lignes, empreinte du fichier) pour pouvoir reproduire un résultat ;
- un **rapport de qualité** avec une **alerte** si le taux de rejet dépasse un seuil.

In [ ]:
import os, datetime
protege_final = etape_protection(v1)
protege_final.to_csv("registre_propre.csv", index=False)
empreinte = hashlib.sha256(open("registre_propre.csv", "rb").read()).hexdigest()
manifeste = {"fichier": "registre_propre.csv", "lignes": len(protege_final),
             "sha256": empreinte[:16], "cree_le": datetime.date.today().isoformat(),
             "etapes": [n for n, _ in ETAPES] + ["validation", "protection"]}
print(json.dumps(manifeste, indent=2, ensure_ascii=False))

taux_rejet = len(q1) / len(sale)
SEUIL_ALERTE = 0.10
print(f"\nTaux de rejet : {taux_rejet:.1%}")
print("🔔 ALERTE : taux de rejet anormal, prévenir le responsable de la donnée" if taux_rejet > SEUIL_ALERTE else "✔ Qualité dans la norme")
print("\nMotifs de rejet les plus fréquents :")
print(q1["motifs"].str.split(",").explode().value_counts().to_string())

## 8 · De ce TP à la production
Ce notebook montre les **idées**. En production, on ajoute :
- **Collecte** : export CSV ou API d'un outil de collecte (Kobo, ODK) ou lecture d'une base ;
- **Orchestration** : exécution planifiée et surveillée (cron, Airflow, Prefect, GitHub Actions…), reprise après incident, alertes ;
- **Tests** : tests unitaires des étapes, contrôles de schéma (colonnes, types) à l'entrée ;
- **Stockage** : brut conservé à part, versions des jeux, accès restreints, durée de conservation ;
- **Traçabilité** : qui a exécuté quoi, avec quel code et quelles données.

## 9 · Bilan
| Étape | Principe | Contrôle |
|---|---|---|
| Profil | Mesurer avant de corriger | Taux de vides, doublons, formes |
| Nettoyage | Étapes pures, rejouables | Idempotence, tests |
| Validation | Règles explicites, quarantaine avec motif | Taux de rejet, motifs |
| Protection | Pseudonymiser, généraliser | k-anonymat, cadre légal |
| Exploitation | Version, journal, alertes | Manifeste, rapport de qualité |

**À retenir**
1. Garder le **brut** intact ; produire les jeux propres par des étapes **rejouables**.
2. **Mesurer** (profil, taux de rejet) plutôt que supposer.
3. **Isoler** les lignes douteuses avec leur motif ; ne jamais supprimer en silence.
4. L'**ordre** des étapes compte (normaliser avant de dédoublonner).
5. Pseudonymisé n'est pas anonyme : minimisation, k-anonymat, accès restreints, cadre légal.

*Pour revenir sur les idées sans code : consultez l'atelier de la version non IT.*